# KiwiLM 3 — train workflow qualification (M2)

## Goal

Test notebook control and checkpoint recovery with a **tiny V2 causal model**. This is not KiwiLM 3 training: the encoder and masked-diffusion objective are later phases. All installation, Drive, training and evaluation actions are **off by default**.

The synthetic corpus and 16-wide model measure infrastructure, not model quality or production throughput.

## Setup

Open in a manually selected Colab CPU/GPU/single-chip TPU runtime. No CLI provisioning is used. Build the reviewed wheel locally with `uv build --wheel`, then upload it to the path below. For TPU use `DEVICE='xla', PRECISION='bf16'`; for a T4 choose `DEVICE='cuda', PRECISION='fp16'`, not BF16. CPU uses FP32. The worker runs in isolated Python 3.12; system/kernel torch is not replaced. See `docs/notebook-workflow.md` for the two-runtime recovery procedure.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

IN_COLAB = Path("/content").is_dir()
WORKSPACE = Path("/content/kiwilm-m2") if IN_COLAB else Path("runs/notebook-m2")
DRIVE_MOUNT = Path("/content/drive")
BACKUP_ROOT = DRIVE_MOUNT / "MyDrive/KiwiLM3" if IN_COLAB else Path("runs/notebook-backups")
WHEEL = BACKUP_ROOT / "wheels" / "kiwilm-0.1.0-py3-none-any.whl"
ENVIRONMENT = Path("/content/kiwilm-m2-env")
WORKER_PYTHON = Path(sys.executable)

DEVICE = "cpu"                    # cpu / cuda / xla — select manually
PRECISION = "fp32"                # CPU fp32; GPU fp16/bf16/fp32; TPU bf16
MOUNT_DRIVE = False
SETUP_ENVIRONMENT = False
PREPARE_DATA = False
RUN_PREFLIGHT = False
START_TRAINING = False
RUN_EVALUATION = False
RESTORE_FOR_EVALUATION = False
EXPORT_SAFETENSORS = False
MODE = "fresh"                    # change to resume after VM replacement
STOP_AFTER_STEP = None     # full locked budget is 8 steps / 256 tokens

print("Actions are disabled until explicitly enabled. No VM allocation or training has started.")

Actions are disabled until explicitly enabled. No VM allocation or training has started.


### 1. Mount Drive and install the worker

Enable these two switches only in Colab. Grant Drive access yourself. Setup can install packages, but never starts training. Use the same reviewed wheel on a replacement runtime; changed code or runtime versions refuse resume.

In [2]:
if MOUNT_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting is only for a manually opened Colab runtime")
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
else:
    print("Drive mount skipped.")

if SETUP_ENVIRONMENT:
    if not IN_COLAB:
        raise RuntimeError("Locally use uv sync --locked --extra notebooks instead")
    if not WHEEL.is_file():
        raise FileNotFoundError("Upload the reviewed wheel and mount Drive first")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(WHEEL)], check=True)
    from kiwilm.notebook_setup import prepare_environment
    WORKER_PYTHON = prepare_environment(WHEEL, ENVIRONMENT, DEVICE)
    print(f"Isolated worker ready: {WORKER_PYTHON}")
else:
    print("Environment installation skipped; local actions use the current installed kiwilm environment.")

Drive mount skipped.
Environment installation skipped; local actions use the current installed kiwilm environment.


## Steps

### 2. Freeze the experiment

Copy this configuration unchanged when resuming or evaluating. Each backup namespace includes its contract digest. Fresh and resume are explicit; there is no warm-start fallback. A new empty local run directory is required on each invocation.

In [3]:
CONFIG = {
    "experiment": "m2-recovery-smoke",
    "device": DEVICE, "precision": PRECISION,
    "max_tokens": 256, "warmup_tokens": 64,
    "batch_size": 2, "grad_accum_steps": 1, "context_length": 16,
    "seed": 42, "noise_seed": 142,
    "checkpoint_interval": 2, "eval_interval": 2, "eval_batches": 2, "lr": 0.001,
}
DATA_DIR = WORKSPACE / "data"
RUN_DIR = WORKSPACE / "run-1"       # change to run-2 if this local directory already has state
REQUEST = {
    "config": CONFIG, "data_dir": str(DATA_DIR), "run_dir": str(RUN_DIR),
    "backup_root": str(BACKUP_ROOT),
    "storage_root": str(DRIVE_MOUNT) if IN_COLAB else None,
}
print(json.dumps({"config": CONFIG, "mode": MODE, "start_training": START_TRAINING}, indent=2))

{
  "config": {
    "experiment": "m2-recovery-smoke",
    "device": "cpu",
    "precision": "fp32",
    "max_tokens": 256,
    "warmup_tokens": 64,
    "batch_size": 2,
    "grad_accum_steps": 1,
    "context_length": 16,
    "seed": 42,
    "noise_seed": 142,
    "checkpoint_interval": 2,
    "eval_interval": 2,
    "eval_batches": 2,
    "lr": 0.001
  },
  "mode": "fresh",
  "start_training": false
}


### 3. Prepare bounded data and check the backend

The local synthetic corpus is deterministic and regenerated in a fresh VM. Preparation does not download SmolLM or overwrite an existing corpus. Preflight checks integrity, precision, backend availability and a small matrix operation. There is no automatic CPU fallback.

In [4]:
if PREPARE_DATA:
    from kiwilm.notebook_setup import run_notebook_action
    prepared = run_notebook_action(WORKER_PYTHON, "prepare", REQUEST)
    print("Prepared fingerprint:", prepared["data"]["fingerprint"])
else:
    print("Data preparation skipped.")

if RUN_PREFLIGHT:
    from kiwilm.notebook_setup import run_notebook_action
    checked = run_notebook_action(WORKER_PYTHON, "preflight", REQUEST)
    print(json.dumps({
        "identity": checked["identity"], "backend": checked["backend"],
        "versions": checked["contract"]["runtime_versions"], "counters": checked["counters"],
    }, indent=2))
else:
    print("Backend preflight skipped.")

Data preparation skipped.
Backend preflight skipped.


### 4. Explicit train or resume

Set `START_TRAINING=True` only when ready. Metrics stream below; publication waits for verified Drive readback and retains latest/previous. Interrupting a worker preserves the last committed boundary, not necessarily its last printed step. With the smoke default, stop at step 4, replace the VM, choose `MODE='resume'` and `STOP_AFTER_STEP=None` to finish the same 256-token schedule.

In [5]:
if START_TRAINING:
    from kiwilm.notebook_setup import run_notebook_action
    trained = run_notebook_action(WORKER_PYTHON, "train", {
        **REQUEST, "mode": MODE, "start_training": True,
        "stop_after_step": STOP_AFTER_STEP,
    })
    print(json.dumps(trained, indent=2))
else:
    print("Training disabled. No optimizer updates were performed.")

Training disabled. No optimizer updates were performed.


## Checks

### 5. Restore, evaluate and optionally export

For a separate evaluation VM set `RESTORE_FOR_EVALUATION=True` with an empty local run directory and the original configuration. Evaluation is CPU/FP32 and reports causal validation, health, cached direct/rollover parity, and a short sample. These are workflow checks, not diffusion or quality benchmarks. Export is inference-only BF16 Safetensors and refuses an existing output; it is not a resume checkpoint.

In [6]:
if RESTORE_FOR_EVALUATION:
    from kiwilm.notebook_setup import run_notebook_action
    restored = run_notebook_action(WORKER_PYTHON, "restore", REQUEST)
    print("Restored committed step:", restored["step"])

if RUN_EVALUATION:
    from kiwilm.notebook_setup import run_notebook_action
    report = run_notebook_action(WORKER_PYTHON, "evaluate", {
        **REQUEST, "checkpoint": str(RUN_DIR / "latest.pt"),
        "export_dir": str(WORKSPACE / "probe-bf16") if EXPORT_SAFETENSORS else None,
    })
    print(json.dumps({key: value for key, value in report.items() if key != "health"}, indent=2))
    print("Per-block health is available in report['health'].")
else:
    print("Evaluation/export skipped; no checkpoint was read or published.")

Evaluation/export skipped; no checkpoint was read or published.


## Next Steps

Record the contract identity, committed step/token counts, worker versions, and parity result for both runtimes. Local CPU continuation is regression-tested; actual Colab GPU/TPU plus Drive recovery must be qualified by running this procedure yourself. Do not infer production TPU performance from the tiny probe. The next model phase adds the bidirectional encoder; masked-diffusion loss/noise and retrieval evaluation follow separately. Do not start a long training run from this M2 qualification notebook.